In [1]:
from pandas import read_csv
from yfinance import download

In [2]:
path = r"Q:\financial_data_pipelines\data\pipelines\b3_indices_segmentos_setoriais\processed\transform_1\IBEP.csv"
df_setor = read_csv(path)
df_setor.tail(3)

,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
73,VIVA3,VIVARA S.A.,ON NM,123.160.591,"0,208"
74,WEGE3,WEG,ON EDJ NM,1.485.954.732,"3,521"
75,YDUQ3,YDUQS PART,ON NM,260.753.144,"0,156"


In [3]:
tickers = [ticker+".SA" for ticker in df_setor['Código']]
df_series = download(tickers, period="10y", interval="1d", auto_adjust=False)

[*********************100%***********************]  76 of 76 completed


In [4]:
df_series.tail(3)

Price      Adj Close                                                     \
Ticker      ABEV3.SA   ALOS3.SA ASAI3.SA AURE3.SA   AXIA3.SA   AXIA6.SA   
Date                                                                      
2025-12-22     13.12  27.379999     7.10    11.80  48.650002  50.830002   
2025-12-23     13.47  28.100000     7.22    11.93  49.950001  52.599998   
2025-12-26     13.72  28.280001     7.26    11.94  49.880001  52.259998   

Price                                             ...   Volume           \
Ticker       AZZA3.SA B3SA3.SA BBDC3.SA BBDC4.SA  ... TOTS3.SA UGPA3.SA   
Date                                              ...                     
2025-12-22  23.400000    13.24    15.70    18.24  ...  3347000  5020300   
2025-12-23  23.820000    13.53    15.85    18.41  ...  1327800  3104000   
2025-12-26  24.459999    13.72    15.86    18.40  ...  1337100  2797000   

Price                                                                       \
Ticker      USIM5.SA  VALE3.SA    VAMO3.SA    VBBR3.SA   VIVA3.SA VIVT3.SA   
Date                                                                         
2025-12-22  10616900  27853000  13145900.0  10010500.0  2700800.0  4055800   
2025-12-23   4044100  17140900  11677300.0   5862000.0  3013100.0  1754900   
2025-12-26   5178300  16368800  14043600.0   5504800.0  2178600.0  1928700   

Price                         
Ticker     WEGE3.SA YDUQ3.SA  
Date                          
2025-12-22  6391700  4494900  
2025-12-23  4886100  2688900  
2025-12-26  1829300  1636200  

[3 rows x 456 columns]

In [5]:
df_adj_close = df_series['Adj Close']
df_adj_close.tail(3)

Ticker,ABEV3.SA,ALOS3.SA,ASAI3.SA,AURE3.SA,AXIA3.SA,AXIA6.SA,AZZA3.SA,B3SA3.SA,BBDC3.SA,BBDC4.SA,...,TOTS3.SA,UGPA3.SA,USIM5.SA,VALE3.SA,VAMO3.SA,VBBR3.SA,VIVA3.SA,VIVT3.SA,WEGE3.SA,YDUQ3.SA
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-22,13.12,27.379999,7.10,11.80,48.650002,50.830002,23.400000,13.24,15.70,18.24,...,42.560001,20.440001,5.88,72.919998,3.10,24.84,32.090000,32.459999,47.660000,11.96
2025-12-23,13.47,28.100000,7.22,11.93,49.950001,52.599998,23.820000,13.53,15.85,18.41,...,42.770000,20.930000,5.90,72.900002,3.19,25.24,33.330002,32.580002,48.279999,12.54
2025-12-26,13.72,28.280001,7.26,11.94,49.880001,52.259998,24.459999,13.72,15.86,18.40,...,42.380001,20.860001,5.92,73.120003,3.27,25.25,33.020000,33.020000,48.750000,12.56


In [6]:
df_retorno = df_adj_close.pct_change(1)
df_metricas = df_retorno.std().to_frame(name='ret_std')
df_metricas['ret_mean'] = df_retorno.mean()
df_metricas

C:\Users\rian\AppData\Local\Temp\ipykernel_16308\3345645728.py:1: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_retorno = df_adj_close.pct_change(1)


,ret_std,ret_mean
Ticker,,
ABEV3.SA,0.016528,0.000201
ALOS3.SA,0.025406,0.000303
ASAI3.SA,0.025497,-0.000205
AURE3.SA,0.015964,0.000056
AXIA3.SA,0.031995,0.001500
...,...,...
VBBR3.SA,0.024600,0.000664
VIVA3.SA,0.030752,0.000766
VIVT3.SA,0.017227,0.000729


In [7]:
from sklearn.preprocessing import StandardScaler

x = df_metricas[['ret_std', 'ret_mean']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(x)

In [8]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

df_metricas['cluster'] = kmeans.fit_predict(X_scaled)

In [9]:
df_metricas['cluster'].value_counts()

cluster
3    30
2    23
0    19
1     4
Name: count, dtype: int64

In [10]:
df_plot = df_metricas.reset_index()
df_plot

,Ticker,ret_std,ret_mean,cluster
0,ABEV3.SA,0.016528,0.000201,3
1,ALOS3.SA,0.025406,0.000303,2
2,ASAI3.SA,0.025497,-0.000205,2
3,AURE3.SA,0.015964,0.000056,3
4,AXIA3.SA,0.031995,0.001500,0
...,...,...,...,...
71,VBBR3.SA,0.024600,0.000664,3
72,VIVA3.SA,0.030752,0.000766,0
73,VIVT3.SA,0.017227,0.000729,3
74,WEGE3.SA,0.021227,0.001145,3


In [11]:
import plotly.express as px

ticker_destacado = 'KLBN11.SA'

fig = px.scatter(
    df_plot,
    x='ret_mean',
    y='ret_std',
    color='cluster',
    hover_name='Ticker',
    custom_data=['cluster'],
    title='Desvio padrão médio x Retorno médio — Agrupamento por Cluster',
    labels={
        'ret_mean': 'Retorno médio',
        'ret_std': 'Desvio padrão (risco)',
        'cluster': 'Cluster'
    },
    template='seaborn'
)

# Eixos com linha zero (referência estatística)
fig.update_xaxes(
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='black'
)

fig.update_yaxes(
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='black'
)

# Pontos mais legíveis
fig.update_traces(
    marker=dict(
        size=7,
        line=dict(width=1, color='black')
    ),
    opacity=1,
    hovertemplate=
        '<b>%{hovertext}</b><br>' +
        'Retorno médio: %{x:.4f}<br>' +
        'Desvio padrão: %{y:.4f}<br>' +
        'Cluster: %{customdata[0]}'
)

# Layout geral
fig.update_layout(
    legend_title_text='Cluster',
    title_x=0.5,
    width=1200,
    height=550
)

# Overlay do ticker destacado
df_highlight = df_plot[df_plot['Ticker'] == ticker_destacado]

fig.add_scatter(
    x=df_highlight['ret_mean'],
    y=df_highlight['ret_std'],
    mode='markers+text',
    marker=dict(
        size=15,
        symbol='star',
        color='yellow',
        line=dict(width=1, color='black')
    ),
    text=df_highlight['Ticker'],
    textposition='top center',
    name=f'Destaque: {ticker_destacado}'
)

fig.show()


In [17]:
df_metricas.loc[df_metricas['cluster'] == 3]

,ret_std,ret_mean,cluster
Ticker,,,
ABEV3.SA,0.016528,0.000201,3
AURE3.SA,0.015964,0.000056,3
B3SA3.SA,0.023534,0.000939,3
BBDC3.SA,0.020826,0.000615,3
BBDC4.SA,0.021822,0.000728,3
CPFE3.SA,0.016356,0.000815,3
EGIE3.SA,0.014120,0.000557,3
EMBJ3.SA,0.018407,0.001033,3
ENGI11.SA,0.018286,0.000857,3


In [19]:
list(df_metricas.loc[df_metricas['cluster'] == 3].index)

['ABEV3.SA',
 'AURE3.SA',
 'B3SA3.SA',
 'BBDC3.SA',
 'BBDC4.SA',
 'CPFE3.SA',
 'EGIE3.SA',
 'EMBJ3.SA',
 'ENGI11.SA',
 'EQTL3.SA',
 'FLRY3.SA',
 'HYPE3.SA',
 'IGTI11.SA',
 'ISAE4.SA',
 'ITSA4.SA',
 'ITUB4.SA',
 'KLBN11.SA',
 'MOTV3.SA',
 'MULT3.SA',
 'PSSA3.SA',
 'RADL3.SA',
 'SANB11.SA',
 'SLCE3.SA',
 'SUZB3.SA',
 'TAEE11.SA',
 'TIMS3.SA',
 'TOTS3.SA',
 'VBBR3.SA',
 'VIVT3.SA',
 'WEGE3.SA']